# World Brain Factory вЂ” free Unsloth training

This notebook trains a **candidate**, not a production model. In Colab choose **Runtime в†’ Change runtime type в†’ GPU**. Free GPU availability is controlled by Colab and is not guaranteed.


In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/mpaykin1/World_server.git'
REF = os.environ.get('WORLD_SERVER_REF', 'master')
ROOT = pathlib.Path('/content/World_server')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', REF, REPO_URL, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'fetch', 'origin', REF, '--depth', '1'])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', '-B', REF, 'FETCH_HEAD'])
print(subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
DATASET_DIR = '/content/world-brain'
subprocess.check_call(['node', str(ROOT/'scripts/world-brain-factory.cjs'), 'prepare', '--out', DATASET_DIR], cwd=str(ROOT))
subprocess.check_call(['node', str(ROOT/'scripts/world-brain-factory.cjs'), 'verify', '--dir', DATASET_DIR], cwd=str(ROOT))


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'unsloth', 'datasets', 'trl'])


In [ ]:
MODEL = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
OUTPUT_DIR = '/content/world-brain-model'
cmd = [sys.executable, str(ROOT/'scripts/train-world-brain-unsloth.py'), '--dataset-dir', DATASET_DIR, '--model', MODEL, '--output-dir', OUTPUT_DIR, '--max-steps', '60', '--export-gguf']
subprocess.check_call(cmd, cwd=str(ROOT))


In [ ]:
print((pathlib.Path(OUTPUT_DIR)/'candidate-manifest.json').read_text())
print((pathlib.Path(OUTPUT_DIR)/'metrics.json').read_text())
print('Candidate artifacts:', [str(p) for p in pathlib.Path(OUTPUT_DIR).iterdir()])
